In [ ]:
!pip install rocketcea cantera

In [ ]:
from pathlib import Path
import rocketcea.cea_obj

path = Path(rocketcea.cea_obj.__file__)

text = path.read_text()

old = 'set_py_cea_line(N," rocket %s  %s"%(eqfrStr,pc_str) + subar_str + pcope_str + eps_str )'
new = 'set_py_cea_line(N," rocket %s ions  %s"%(eqfrStr,pc_str) + subar_str + pcope_str + eps_str )'

if old not in text:
    print("Target line NOT found — nothing changed.")
else:
    # Make a backup first
    backup = path.with_suffix(".py.backup")
    backup.write_text(text)

    # Patch it
    path.write_text(text.replace(old, new, 1))

    print("Patched:", path)
    print("Backup:", backup)

In [ ]:
from rocketcea.cea_obj import CEA_Obj
from rocketcea import cea_obj

card_str = """
fuel MyCustomPropellant C 6 N 32 O 6 F 21 H 9  wt=100.0
h,kj/mol=3050  t(k)=298.15
"""

cea_obj.add_new_fuel("MyCustomPropellant", card_str)

ispObj = CEA_Obj(propName="MyCustomPropellant")

Pc = 1000.0
eps = 100.0

final_text = ispObj.get_full_cea_output(
    Pc=Pc,
    eps=eps
)

_, _, _, mw_exhaust, gamma = ispObj.get_IvacCstrTc_ChmMwGam(
    Pc=Pc,
    eps=eps
)

with open("propellant_products.txt", "w") as f:
    f.write(final_text)

print(f"Exhaust MW: {mw_exhaust}, Gamma: {gamma}")

In [ ]:
!cat /kaggle/working/propellant_products.txt

In [3]:
%%bash

cat > propellant.yaml << 'EOF'
description: |
  Equilibrium model for C6N32O6F21H9 monopropellant.
  Custom reactant enthalpy from bond-additivity estimate.
  Product species imported from Cantera's nasa_gas.yaml database.

phases:
- name: gas
  thermo: ideal-gas

  elements: [C, N, O, F, H]

  species:
  - species: [MyCustomPropellant]

  - nasa_gas.yaml/species:
    - H
    - H2
    - O
    - O2
    - OH
    - H2O
    - N
    - N2
    - NO
    - F
    - F2
    - HF
    - C
    - CO
    - CO2
    - CN
    - HCN
    - CF
    - CF2
    - CF3
    - CF4
    - COF2
    - COF
    - FCN
    - NF

  kinetics: none

  state:
    T: 298.15 K
    P: 1 atm

species:

- name: MyCustomPropellant

  composition:
    C: 6
    N: 32
    O: 6
    F: 21
    H: 9

  thermo:
    model: constant-cp
    T0: 298.15 K

    # 2650 matches the CEA run. Use 2250 or 3050 for the low and high cases
    h0: 3050 kJ/mol

    # Placeholder, same 74 atoms as before
    s0: 1224.5588 J/mol/K  
EOF

In [ ]:
import cantera as ct

species = ct.Species.list_from_file("nasa_gas.yaml")
names = {sp.name for sp in species}

wanted = [
    "H", "H2", "O", "O2", "OH", "H2O",
    "N", "N2", "NO",
    "F", "F2", "HF",
    "C", "CO", "CO2", "CN", "HCN"
]

for name in wanted:
    print(f"{name:5s}", "OK" if name in names else "MISSING")

In [5]:
import sys
import contextlib
import cantera as ct


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, s):
        for st in self.streams:
            st.write(s)
        return len(s)

    def flush(self):
        for st in self.streams:
            st.flush()


OUT = "/kaggle/working/cantera_output.txt"

with open(OUT, "w") as f, contextlib.redirect_stdout(Tee(sys.stdout, f)):
    gas = ct.Solution("/kaggle/working/propellant.yaml")

    T0 = 298.15
    Pc = 1000.0 * 6894.757293168

    gas.TPX = T0, Pc, {"MyCustomPropellant": 1.0}
    print("Initial state")
    print(gas.report())

    gas.equilibrate(
        "HP",
        solver="vcs",
        max_steps=100000000,
        max_iter=1000000,
        estimate_equil=-1,
        log_level=1
    )

    print("Equilibrium state")
    print(gas.report(threshold=0.0))
    print("Chamber temperature K", gas.T)
    print("Chamber pressure Pa", gas.P)
    print("Mean molecular weight", gas.mean_molecular_weight)
    print("Enthalpy J per kg", gas.enthalpy_mass)
    print("Entropy J per kg per K", gas.entropy_mass)
    print("Mole fractions")
    for name, x in sorted(zip(gas.species_names, gas.X), key=lambda p: -p[1]):
        print(name, x)

print("Saved to", OUT)

Initial state

  gas:

       temperature   298.15 K
          pressure   6.8948e+06 Pa
           density   2849 kg/m^3
  mean mol. weight   1024.3 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy        2.9776e+06          3.05e+09  J
   internal energy        2.9752e+06        3.0475e+09  J
           entropy            1161.2        1.1895e+06  J/K
    Gibbs function        2.6314e+06        2.6954e+09  J
 heat capacity c_p                 0                 0  J/K
 heat capacity c_v            -8.117           -8314.5  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
MyCustomPropellant                 1                 1            1087.3
     [  +25 minor]                 0                 0  

Trying VCS equilibrium solver
VCS solver succeeded
Equilibrium state

  gas